# WP3b: SAR Ice Classification — Hornsund Method (SAR Track)

**Owner: Julian**

Implements the GLCM + SVM pipeline from Williams & Swirad (2025) adapted for Sermilik Fjord.

**Pipeline:**
1. Build 10-band GLCM composite (from `02b`)
2. Sample training pixels from manually labelled polygons
3. Train SVM (RBF kernel)
4. Classify → multi-class ice-type map
5. Collapse to binary `ice_s1` for time series fusion

**Ice classes:**

| Value | Class |
|---|---|
| 0 | Open water |
| 1 | Drift ice |
| 2 | Landfast ice |
| 3 | Glacier ice |

> **Dependency:** training polygons from the S2 track (`03a`) — get the GEE asset path from your colleague.

In [ ]:
import ee
import geemap
import sys
sys.path.insert(0, '..')
from src.utils import load_aoi, get_gee_project
from src.preprocessing_s1 import filter_dual_pol, preprocess_s1, COMPOSITE_BANDS
from src.classification_s1 import train_svm, classify_svm, to_binary

ee.Initialize(project=get_gee_project())
aoi = load_aoi()

## 3b.1 Build 10-band GLCM composite

Recreates the preprocessed S1 collection (same as `02b`). GEE is lazy — this does not
re-run computation unless you call `.getInfo()` or export.

In [ ]:
START, END = '2019-01-01', '2024-12-31'

s1 = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate(START, END)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'HH'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'HV'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select(['HH', 'HV'])
)
s1 = filter_dual_pol(s1).map(preprocess_s1)
print('S1 images with GLCM composites:', s1.size().getInfo())
print('Bands:', s1.first().bandNames().getInfo())

## 3b.2 Training data

Load the training FeatureCollection digitised in `03a_classification_s2.ipynb`.
Polygons should cover all four ice classes across both winter and summer scenes.

**How to create training data:**
1. Open a winter S2 scene (e.g. January) alongside the S1 HH layer in GEE
2. Digitise polygons for each class; add `class` property (0–3)
3. Do the same for a summer scene (June–August)
4. Upload merged FeatureCollection to GEE as an asset
5. Paste the asset path below

In [ ]:
# TODO: replace with your GEE asset path
# TRAINING_ASSET = 'projects/<your-project>/assets/sermilik_training_polygons'
#
# training_fc = ee.FeatureCollection(TRAINING_ASSET)
#
# Sample training pixels from the composite bands
# training_pixels = s1.first().select(COMPOSITE_BANDS).sampleRegions(
#     collection=training_fc,
#     properties=['class'],
#     scale=50,
#     tileScale=4,
# )
# print('Training samples:', training_pixels.size().getInfo())
print('Training data: upload polygons and uncomment the block above.')

## 3b.3 Train SVM

RBF kernel with default gamma=0.5, cost=10. Tune via cross-validation once training data is ready.

In [ ]:
# TODO: uncomment after training data is set up
# classifier = train_svm(training_pixels)
print('SVM training: complete training data step first.')

## 3b.4 Classify + binary collapse

Produces two output bands per image:
- `ice_type_s1`: multi-class (0–3)
- `ice_s1`: binary ice / water (for time series fusion)

In [ ]:
# TODO: uncomment after SVM is trained
# s1_classified = s1.map(lambda img: to_binary(classify_svm(img, classifier)))
#
# sample = s1_classified.first().clip(aoi)
# Map = geemap.Map()
# Map.centerObject(aoi, zoom=9)
# Map.addLayer(
#     sample.select('ice_type_s1'),
#     {'min': 0, 'max': 3, 'palette': ['#1a6faf', '#a8d8ea', 'white', '#b0c4de']},
#     'Ice type (SVM)'
# )
# Map.addLayer(
#     sample.select('ice_s1'),
#     {'min': 0, 'max': 1, 'palette': ['#1a6faf', 'white']},
#     'Ice binary (S1)'
# )
# Map
print('Classification: complete SVM training step first.')

## Notes

- 40/60 train/test split is recommended (Williams & Swirad 2025). Stratify by season.
- The geometric reclassification step from the paper (object orientation/roundness) cannot
  be done in GEE — it is deferred to a future Python post-processing notebook using rasterio + shapely.
- The `ice_s1` binary band is the output consumed by the fusion notebook.